# 069 — Modelos visión-lenguaje

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**CLIP (2021):** dos codificadores — imagen `f` y texto `g` — proyectan a un espacio común
con embeddings normalizados; la similitud coseno es un producto punto. Se entrena con
**aprendizaje contrastivo** (InfoNCE simétrica) sobre lotes de N pares imagen-texto de la
web (400 M): la matriz N×N de similitudes debe tener la diagonal alta. La **temperatura τ**
escala los logits: no cambia el ranking, cambia la confianza aparente.

**Zero-shot:** cada clase se convierte en un prompt ("una foto de un {perro}"), se codifica
con `g`, y la imagen se asigna a la clase de mayor coseno. Zero-shot ≠ sin datos: significa
sin *fine-tuning* por tarea.

**Límites:** embedding global (sabe *qué*, no *dónde* → sin grounding fino), comportamiento
de bolsa de palabras ("perro persigue gato" ≈ "gato persigue perro"), débil en conteo,
vulnerable a **ataques tipográficos** (texto impreso dentro de la imagen). Los VLM
generativos (Flamingo, LLaVA) conectan un encoder visual a un LLM para VQA y descripción
libre — heredando sus alucinaciones.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Cosenos (vectores ya normalizados → producto punto):
`s(img, gato) = 0.8`, `s(img, perro) = 0.6`, `s(img, avión) = 0.0`. Zero-shot → **gato**.

**Ejercicio 2.** Con τ = 0.5, logits (1.6, 1.2, 0): exp = (4.95, 3.32, 1.00), suma 9.27 →
`p = (0.53, 0.36, 0.11)`. Con τ = 0.1, logits (8, 6, 0): exp ≈ (2981, 403, 1) →
`p ≈ (0.881, 0.119, 0.000)`. La clase no cambia (el ranking es invariante a τ); cambia la
**calibración**: τ pequeña produce confianza mucho más afilada con la misma evidencia.

**Ejercicio 3.** (a) P. ej. "una radiografía de tórax con neumonía" / "una radiografía de
tórax normal". Aun con buenos prompts, las radiografías apenas aparecen entre los 400 M de
pares web con descripciones diagnósticas fiables: el zero-shot se degrada fuera de la
distribución de entrenamiento — dominio especializado exige evaluación propia o
fine-tuning. (b) "el perro muerde al hombre" / "el hombre muerde al perro": mismas
palabras, roles invertidos; el comportamiento de bolsa de palabras de CLIP los acerca en el
espacio aunque el significado sea opuesto.

**Ejercicio 4.** Implementación debajo: reproduce (0.8, 0.6, 0.0) y las probabilidades de
la solución 2.


In [ ]:
result = run_lab("attention", seed=69)
assert result["kind"] == "attention"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 2 — verificados con código
import math

def coseno(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb)

def softmax_temp(sims, tau):
    exps = [math.exp(s / tau) for s in sims]
    total = sum(exps)
    return [round(e / total, 3) for e in exps]

img = (0.6, 0.8, 0)
textos = {"gato": (0, 1, 0), "perro": (1, 0, 0), "avión": (0, 0, 1)}
sims = [coseno(img, t) for t in textos.values()]
print("similitudes:", dict(zip(textos, [round(s, 3) for s in sims])))
print("tau=0.5:", dict(zip(textos, softmax_temp(sims, 0.5))))
print("tau=0.1:", dict(zip(textos, softmax_temp(sims, 0.1))))
print("zero-shot:", max(textos, key=lambda k: coseno(img, textos[k])))


## Reflexión

1. Tu clasificador zero-shot funciona con fotos de catálogo pero falla con fotos tomadas
   por usuarios. ¿Qué diferencia de distribución lo explica y qué harías **sin** reentrenar
   CLIP (prompts, preprocesado, umbrales)?
2. Un producto trae impresa la palabra "banana" y CLIP lo clasifica como banana. ¿Cómo se
   llama este fallo, por qué ocurre en un espacio compartido imagen-texto y qué mitigación
   propondrías?
3. Si la clase verdadera no está en tu lista de prompts, ¿qué hace el argmax igualmente?
   ¿Cómo diseñarías una opción "ninguna de las anteriores" con umbrales de similitud?
